In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
station_day_data = '/content/drive/My Drive/impact_metrics/station_day.csv'
station_metadata = '/content/drive/My Drive/impact_metrics/stations.csv'

In [ ]:
df = pd.read_csv(station_day_data)
df_meta = pd.read_csv(station_metadata)
df = pd.merge(df, df_meta, on='StationId', how='left')
df['Date'] = pd.to_datetime(df['Date'])
df = df.set_index('Date').sort_index()

df = df.sort_values(by = ['City', 'StationId', 'Date'])

imp_cols = ['PM2.5', 'PM10', 'NO',	'NO2',	'NOx',	'NH3',	'CO',	'SO2',	'O3',	'Benzene',	'Toluene',	'Xylene']

In [ ]:
df

,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,StationName,City,State,Status
Date,,,,,,,,,,,,,,,,,,,
2015-01-01,GJ001,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN,"Maninagar, Ahmedabad - GPCB",Ahmedabad,Gujarat,Active
2015-01-02,GJ001,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN,"Maninagar, Ahmedabad - GPCB",Ahmedabad,Gujarat,Active
2015-01-03,GJ001,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN,"Maninagar, Ahmedabad - GPCB",Ahmedabad,Gujarat,Active
2015-01-04,GJ001,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN,"Maninagar, Ahmedabad - GPCB",Ahmedabad,Gujarat,Active
2015-01-05,GJ001,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN,"Maninagar, Ahmedabad - GPCB",Ahmedabad,Gujarat,Active
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-27,AP005,15.02,50.94,7.68,25.06,19.54,12.47,0.47,8.55,23.30,2.24,12.07,0.73,41.0,Good,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active
2020-06-28,AP005,24.38,74.09,3.42,26.06,16.53,11.99,0.52,12.72,30.14,0.74,2.21,0.38,70.0,Satisfactory,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active
2020-06-29,AP005,22.91,65.73,3.45,29.53,18.33,10.71,0.48,8.42,30.96,0.01,0.01,0.00,68.0,Satisfactory,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active


In [ ]:
df_imp = df.copy()
print(f"Number of NaNs before imputation: \n{df_imp[imp_cols].isna().sum()}")
for col in imp_cols:
  df_imp[col] = df_imp.groupby('StationId')[col].ffill(limit=2)
  df_imp[col] = df_imp.groupby('StationId')[col].bfill(limit=2)
print(f"Number of NaNs after FIRST LAYER imputation: \n{df_imp[imp_cols].isna().sum()}")

Number of NaNs before imputation: 
PM2.5      21625
PM10       42706
NO         17106
NO2        16547
NOx        15500
NH3        48105
CO         12998
SO2        25204
O3         25568
Benzene    31455
Toluene    38702
Xylene     85137
dtype: int64
Number of NaNs after FIRST LAYER imputation: 
PM2.5      18992
PM10       40710
NO         13761
NO2        13298
NOx        12948
NH3        46284
CO          9528
SO2        22300
O3         22876
Benzene    28907
Toluene    35925
Xylene     84324
dtype: int64


In [ ]:
df_imp['month'] = df_imp.index.to_period('M')

In [ ]:
station_monthly_med = df_imp.groupby(['StationId', 'month'])[imp_cols].median().reset_index()

In [ ]:
for col in imp_cols:
  median_map = station_monthly_med.set_index(['StationId', 'month'])[col]
  flag_col = f'{col}_imputed_II'

  df_imp[flag_col] = df_imp.groupby(['StationId', 'month'])[col].transform(
      lambda x: x.isna() & pd.notna(median_map.loc[x.name[0], x.name[1]])
  )

  df_imp[col] = df_imp.groupby(['StationId', 'month'])[col].transform(
      lambda x: x.fillna(median_map.loc[x.name[0], x.name[1]]) if x.name in median_map.index else x
  )
# df_imp = df_imp.drop(columns = ['month'])

In [ ]:
# logs
print(f"Number of NaNs after SECOND LAYER imputation: \n{df_imp[imp_cols].isna().sum()}")


step_b_imputed_rows = df_imp[[col for col in df_imp.columns if '_imputed_B' in col]].any(axis=1)

stations_imputed_in_step_B = df_imp[step_b_imputed_rows]['StationId'].unique()

# print("The unique stationId(s) for which the Station Monthly Median (Step II) imputation was applied:")
# print(stations_imputed_in_step_B)

# df_imp = df_imp.drop(columns=[col for col in df_imp.columns if '_imputed_B' in col])

Number of NaNs after SECOND LAYER imputation: 
PM2.5      15357
PM10       38344
NO          9932
NO2         9469
NOx        10393
NH3        44470
CO          6613
SO2        18829
O3         19459
Benzene    25832
Toluene    32904
Xylene     83208
dtype: int64


In [ ]:
city_monthly_med = df_imp.groupby(['City', 'month'])[imp_cols].median().reset_index()

In [ ]:
for col in imp_cols:
  median_map = city_monthly_med.set_index(['City', 'month'])[col]
  df_imp[col] = df_imp.groupby(['City', 'month'])[col].transform(
      lambda x: x.fillna(median_map.loc[x.name[0], x.name[1]]) if x.name in median_map.index else x
  )


In [ ]:
print(f"Number of NaNs after THIRD LAYER imputation: \n{df_imp[imp_cols].isna().sum()}")

Number of NaNs after SECOND LAYER imputation: 
PM2.5       3303
PM10       17342
NO          1964
NO2         1994
NOx         3154
NH3        11855
CO           640
SO2         2276
O3          2423
Benzene     4292
Toluene    10711
Xylene     48314
dtype: int64


In [ ]:
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

df_imp['Season'] = df_imp.index.month.map(get_season)

city_seasonal_median = df_imp.groupby(['City', 'Season'])[imp_cols].median().reset_index()

for col in imp_cols:
    median_map = city_seasonal_median.set_index(['City', 'Season'])[col]

    df_imp[col] = df_imp.groupby(['City', 'Season'])[col].transform(
        lambda x: x.fillna(median_map.loc[x.name[0], x.name[1]]) if x.name in median_map.index else x
    )

In [ ]:
df_imp

,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,...,NO2_imputed_II,NOx_imputed_II,NH3_imputed_II,CO_imputed_II,SO2_imputed_II,O3_imputed_II,Benzene_imputed_II,Toluene_imputed_II,Xylene_imputed_II,Season
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-01,GJ001,76.54,118.44,0.92,18.22,17.15,NaN,0.92,27.64,133.36,...,False,False,False,False,False,False,False,False,False,Winter
2015-01-02,GJ001,76.54,118.44,0.97,15.69,16.46,NaN,0.97,24.55,34.06,...,False,False,False,False,False,False,False,False,False,Winter
2015-01-03,GJ001,76.54,118.44,17.40,19.30,29.70,NaN,17.40,29.07,30.70,...,False,False,False,False,False,False,False,False,False,Winter
2015-01-04,GJ001,76.54,118.44,1.70,18.48,17.97,NaN,1.70,18.59,36.08,...,False,False,False,False,False,False,False,False,False,Winter
2015-01-05,GJ001,76.54,118.44,22.10,21.42,37.76,NaN,22.10,39.33,39.31,...,False,False,False,False,False,False,False,False,False,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-27,AP005,15.02,50.94,7.68,25.06,19.54,12.47,0.47,8.55,23.30,...,False,False,False,False,False,False,False,False,False,Summer
2020-06-28,AP005,24.38,74.09,3.42,26.06,16.53,11.99,0.52,12.72,30.14,...,False,False,False,False,False,False,False,False,False,Summer
2020-06-29,AP005,22.91,65.73,3.45,29.53,18.33,10.71,0.48,8.42,30.96,...,False,False,False,False,False,False,False,False,False,Summer


In [ ]:
print(f"Number of NaNs after FOURTH LAYER imputation: \n{df_imp[imp_cols].isna().sum()}")

Number of NaNs after FOURTH LAYER imputation: 
PM2.5          0
PM10        6554
NO             0
NO2            0
NOx         1169
NH3         3707
CO             0
SO2            0
O3           162
Benzene     2872
Toluene     4150
Xylene     34879
dtype: int64


In [ ]:
df_imp['flag_long_missing_data'] = np.where(df_imp[imp_cols].isna().any(axis=1), 1, 0)

df_final = df_imp.dropna(subset=imp_cols)

print(f"Original Data Shape: {df.shape}")
print(f"Final Imputed Data Shape (after dropping extreme NaNs): {df_final.shape}")
print(f"Number of rows flagged for long missing data: {df_imp['flag_long_missing_data'].sum()}")

Original Data Shape: (108035, 19)
Final Imputed Data Shape (after dropping extreme NaNs): (70530, 34)
Number of rows flagged for long missing data: 37505


In [ ]:
df_final = df_final.drop(columns=[col for col in df_final.columns if '_imputed_II' in col])

In [ ]:
df_final

,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,...,Xylene,AQI,AQI_Bucket,StationName,City,State,Status,month,Season,flag_long_missing_data
Date,,,,,,,,,,,,,,,,,,,,,
2017-11-24,AP001,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,...,0.10,NaN,NaN,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11,Autumn,0
2017-11-25,AP001,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,...,0.06,184.0,Moderate,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11,Autumn,0
2017-11-26,AP001,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,...,0.08,197.0,Moderate,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11,Autumn,0
2017-11-27,AP001,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,...,0.12,198.0,Moderate,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11,Autumn,0
2017-11-28,AP001,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,...,0.07,188.0,Moderate,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11,Autumn,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-27,AP005,15.02,50.94,7.68,25.06,19.54,12.47,0.47,8.55,23.30,...,0.73,41.0,Good,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active,2020-06,Summer,0
2020-06-28,AP005,24.38,74.09,3.42,26.06,16.53,11.99,0.52,12.72,30.14,...,0.38,70.0,Satisfactory,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active,2020-06,Summer,0
2020-06-29,AP005,22.91,65.73,3.45,29.53,18.33,10.71,0.48,8.42,30.96,...,0.00,68.0,Satisfactory,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active,2020-06,Summer,0


In [ ]:
df_final['AQI_Bucket'].unique()

array([nan, 'Moderate', 'Poor', 'Very Poor', 'Satisfactory', 'Good',
       'Severe'], dtype=object)

In [ ]:
#CO is in mg/m3; all others are in ug/m3.
AQI_BREAKPOINTS = {
    (0, 50): [  # Good
        (0, 30.0, 'PM2.5'), (0, 50.0, 'PM10'), (0, 40.0, 'NO2'), (0, 60.0, 'O3'),
        (0, 1.0, 'CO'), (0, 40.0, 'SO2'), (0, 200.0, 'NH3'), (0, 5.0, 'Benzene'),
        (0, 14.0, 'Toluene'), (0, 17.0, 'Xylene')
    ],
    (51, 100): [ # Satisfactory
        (31.0, 60.0, 'PM2.5'), (51.0, 100.0, 'PM10'), (41.0, 80.0, 'NO2'), (61.0, 100.0, 'O3'),
        (1.1, 2.0, 'CO'), (41.0, 80.0, 'SO2'), (201.0, 400.0, 'NH3'), (5.1, 10.0, 'Benzene'),
        (14.1, 40.0, 'Toluene'), (17.1, 40.0, 'Xylene')
    ],
    (101, 200): [ # Moderate
        (61.0, 90.0, 'PM2.5'), (101.0, 250.0, 'PM10'), (81.0, 180.0, 'NO2'), (101.0, 168.0, 'O3'),
        (2.1, 10.0, 'CO'), (81.0, 380.0, 'SO2'), (401.0, 800.0, 'NH3'), (10.1, 17.0, 'Benzene'),
        (40.1, 80.0, 'Toluene'), (40.1, 80.0, 'Xylene')
    ],
    (201, 300): [ # Poor
        (91.0, 120.0, 'PM2.5'), (251.0, 350.0, 'PM10'), (181.0, 280.0, 'NO2'), (169.0, 208.0, 'O3'),
        (10.1, 17.0, 'CO'), (381.0, 800.0, 'SO2'), (801.0, 1200.0, 'NH3'), (17.1, 34.0, 'Benzene'),
        (80.1, 120.0, 'Toluene'), (80.1, 120.0, 'Xylene')
    ],
    (301, 400): [ # Very Poor
        (121.0, 250.0, 'PM2.5'), (351.0, 430.0, 'PM10'), (281.0, 400.0, 'NO2'), (209.0, 748.0, 'O3'),
        (17.1, 34.0, 'CO'), (801.0, 1600.0, 'SO2'), (1201.0, 1800.0, 'NH3'), (34.1, 50.0, 'Benzene'),
        (120.1, 240.0, 'Toluene'), (120.1, 240.0, 'Xylene')
    ],
    (401, 500): [ # Severe
        (251.0, 9999.0, 'PM2.5'), (431.0, 9999.0, 'PM10'), (401.0, 9999.0, 'NO2'), (749.0, 9999.0, 'O3'),
        (34.1, 9999.0, 'CO'), (1601.0, 9999.0, 'SO2'), (1801.0, 9999.0, 'NH3'), (50.1, 9999.0, 'Benzene'),
        (240.1, 9999.0, 'Toluene'), (240.1, 9999.0, 'Xylene')
    ]
}

In [ ]:
def get_pollutant_index(pollutant_name):
    pollutant_order = ['PM2.5', 'PM10',	'NO2',	'NH3',	'CO',	'SO2',	'O3',	'Benzene',	'Toluene',	'Xylene']
    try:
        return pollutant_order.index(pollutant_name)
    except ValueError:
        return -1

def calculate_sub_index(pollutant_conc, pollutant_name, breakpoints):
    if pd.isna(pollutant_conc) or pollutant_conc < 0:
        return np.nan

    map_index = get_pollutant_index(pollutant_name)
    if map_index == -1:
        return np.nan

    for (I_L, I_H), pollutant_ranges in breakpoints.items():
        try:
            B_L, B_H, _ = pollutant_ranges[map_index]
        except IndexError:
            continue

        if pollutant_conc >= B_L and pollutant_conc <= B_H:
            # Ip = [(I_H - I_L) / (B_H - B_L)] * (C_p - B_L) + I_L
            if B_H == B_L:
                 return I_H

            Ip = ((I_H - I_L) / (B_H - B_L)) * (pollutant_conc - B_L) + I_L
            return Ip

        if I_H == 500 and pollutant_conc > B_H:
            return 500.0

    return np.nan

def determine_aqi_bucket(aqi_value):
    if pd.isna(aqi_value):
        return np.nan
    elif aqi_value <= 50:
        return 'Good'
    elif aqi_value <= 100:
        return 'Satisfactory'
    elif aqi_value <= 200:
        return 'Moderate'
    elif aqi_value <= 300:
        return 'Poor'
    elif aqi_value <= 400:
        return 'Very Poor'
    else:
        return 'Severe'

def calculate_overall_aqi_and_bucket(row, breakpoints):
    sub_indices = []

    pollutants_to_check = ['PM2.5', 'PM10',	'NO2',	'NH3',	'CO',	'SO2',	'O3',	'Benzene',	'Toluene',	'Xylene']

    for p_name in pollutants_to_check:
        p_conc = row.get(p_name)
        Ip = calculate_sub_index(p_conc, p_name, breakpoints)
        if not pd.isna(Ip):
            sub_indices.append(Ip)

    if not sub_indices:
        return np.nan, np.nan

    final_aqi = max(sub_indices)
    aqi_bucket = determine_aqi_bucket(final_aqi)

    return final_aqi, aqi_bucket

In [ ]:
df_final = df_final.drop(columns=['AQI', 'AQI_Bucket'], errors='ignore')

df_final[['AQI', 'AQI_Bucket']] = df_final.apply(
    lambda row: calculate_overall_aqi_and_bucket(row, AQI_BREAKPOINTS),
    axis=1,
    result_type='expand'
)

In [ ]:
df_final

,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,...,Xylene,StationName,City,State,Status,month,Season,flag_long_missing_data,AQI,AQI_Bucket
Date,,,,,,,,,,,,,,,,,,,,,
2017-11-24,AP001,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,...,0.10,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11,Autumn,0,136.366897,Moderate
2017-11-25,AP001,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,...,0.06,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11,Autumn,0,170.641379,Moderate
2017-11-26,AP001,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,...,0.08,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11,Autumn,0,160.126897,Moderate
2017-11-27,AP001,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,...,0.12,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11,Autumn,0,195.766897,Moderate
2017-11-28,AP001,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,...,0.07,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active,2017-11,Autumn,0,111.855862,Moderate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-27,AP005,15.02,50.94,7.68,25.06,19.54,12.47,0.47,8.55,23.30,...,0.73,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active,2020-06,Summer,0,43.107143,Good
2020-06-28,AP005,24.38,74.09,3.42,26.06,16.53,11.99,0.52,12.72,30.14,...,0.38,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active,2020-06,Summer,0,74.090000,Satisfactory
2020-06-29,AP005,22.91,65.73,3.45,29.53,18.33,10.71,0.48,8.42,30.96,...,0.00,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active,2020-06,Summer,0,65.730000,Satisfactory


In [ ]:
df_final.to_csv('/content/drive/MyDrive/impact_metrics/station_day_imp.csv', index = True)

# CITY DAILY

In [ ]:
df_city = pd.read_csv('/content/drive/MyDrive/impact_metrics/city_day.csv')

In [ ]:
df_city

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29526,Visakhapatnam,2020-06-27,15.02,50.94,7.68,25.06,19.54,12.47,0.47,8.55,23.30,2.24,12.07,0.73,41.0,Good
29527,Visakhapatnam,2020-06-28,24.38,74.09,3.42,26.06,16.53,11.99,0.52,12.72,30.14,0.74,2.21,0.38,70.0,Satisfactory
29528,Visakhapatnam,2020-06-29,22.91,65.73,3.45,29.53,18.33,10.71,0.48,8.42,30.96,0.01,0.01,0.00,68.0,Satisfactory
29529,Visakhapatnam,2020-06-30,16.64,49.97,4.05,29.26,18.80,10.03,0.52,9.84,28.30,0.00,0.00,0.00,54.0,Satisfactory


In [ ]:
df_city['Date'] = pd.to_datetime(df_city['Date'])
df_city = df_city.set_index('Date').sort_index()

df_city = df_city.sort_values(by = ['City', 'Date'])

imp_cols = ['PM2.5', 'PM10', 'NO',	'NO2',	'NOx',	'NH3',	'CO',	'SO2',	'O3',	'Benzene',	'Toluene',	'Xylene']

In [ ]:
df_city_imp = df_city.copy()
print(f"Number of NaNs before imputation: \n{df_city_imp[imp_cols].isna().sum()}")
for col in imp_cols:
  df_city_imp[col] = df_city_imp.groupby('City')[col].ffill(limit=5)
  df_city_imp[col] = df_city_imp.groupby('City')[col].bfill(limit=5)
print(f"Number of NaNs after FIRST LAYER imputation: \n{df_city_imp[imp_cols].isna().sum()}")

Number of NaNs before imputation: 
PM2.5       4598
PM10       11140
NO          3582
NO2         3585
NOx         4185
NH3        10328
CO          2059
SO2         3854
O3          4022
Benzene     5623
Toluene     8041
Xylene     18109
dtype: int64
Number of NaNs after FIRST LAYER imputation: 
PM2.5       3686
PM10       10157
NO          2534
NO2         2549
NOx         3483
NH3         9386
CO          1073
SO2         2628
O3          2725
Benzene     4521
Toluene     7113
Xylene     17432
dtype: int64


In [ ]:
df_city_imp['month'] = df_city_imp.index.to_period('M')

In [ ]:
city_monthly_med = df_city_imp.groupby(['City', 'month'])[imp_cols].median().reset_index()

In [ ]:
for col in imp_cols:
  median_map = city_monthly_med.set_index(['City', 'month'])[col]
  flag_col = f'{col}_imputed_II'

  df_city_imp[flag_col] = df_city_imp.groupby(['City', 'month'])[col].transform(
      lambda x: x.isna() & pd.notna(median_map.loc[x.name[0], x.name[1]])
  )

  df_city_imp[col] = df_city_imp.groupby(['City', 'month'])[col].transform(
      lambda x: x.fillna(median_map.loc[x.name[0], x.name[1]]) if x.name in median_map.index else x
  )

In [ ]:
# logs
print(f"Number of NaNs after SECOND LAYER imputation: \n{df_city_imp[imp_cols].isna().sum()}")


step_b_imputed_rows = df_city_imp[[col for col in df_city_imp.columns if '_imputed_B' in col]].any(axis=1)

cities_imputed_in_step_B = df_city_imp[step_b_imputed_rows]['City'].unique()

Number of NaNs after SECOND LAYER imputation: 
PM2.5       3066
PM10        9539
NO          1903
NO2         1933
NOx         2910
NH3         8810
CO           639
SO2         1934
O3          1991
Benzene     3881
Toluene     6622
Xylene     16912
dtype: int64


In [ ]:
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else: return 'Autumn'

df_city_imp['Season'] = df_city_imp.index.month.map(get_season)

city_seasonal_median = df_city_imp.groupby(['City', 'Season'])[imp_cols].median().reset_index()

for col in imp_cols:
    median_map = city_seasonal_median.set_index(['City', 'Season'])[col]

    df_city_imp[col] = df_city_imp.groupby(['City', 'Season'])[col].transform(
        lambda x: x.fillna(median_map.loc[x.name[0], x.name[1]]) if x.name in median_map.index else x
    )

In [ ]:
print(f"Number of NaNs after THIRD LAYER imputation: \n{df_city_imp[imp_cols].isna().sum()}")

Number of NaNs after THIRD LAYER imputation: 
PM2.5          0
PM10        2464
NO             0
NO2            0
NOx         1169
NH3         2832
CO             0
SO2            0
O3           162
Benzene     2732
Toluene     4010
Xylene     13415
dtype: int64


In [ ]:
#fill zero in all others
df_city_final = df_city_imp.fillna(0)

In [ ]:
df_city_final = df_city_final.drop(columns=[col for col in df_city_final.columns if '_imputed_II' in col])

In [ ]:
# aqi and aqi bucket
df_city_final = df_city_final.drop(columns=['AQI', 'AQI_Bucket'], errors='ignore')

df_city_final[['AQI', 'AQI_Bucket']] = df_city_final.apply(
    lambda row: calculate_overall_aqi_and_bucket(row, AQI_BREAKPOINTS),
    axis=1,
    result_type='expand'
)

In [ ]:
df_city_final

,City,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,month,Season,AQI,AQI_Bucket
Date,,,,,,,,,,,,,,,,,
2015-01-01,Ahmedabad,73.24,118.44,0.92,18.22,17.15,0.00,0.92,27.64,133.36,0.00,0.02,0.00,2015-01,Winter,142.784828,Moderate
2015-01-02,Ahmedabad,73.24,118.44,0.97,15.69,16.46,0.00,0.97,24.55,34.06,3.68,5.50,3.77,2015-01,Winter,142.784828,Moderate
2015-01-03,Ahmedabad,73.24,118.44,17.40,19.30,29.70,0.00,17.40,29.07,30.70,6.80,16.40,2.25,2015-01,Winter,302.757396,Very Poor
2015-01-04,Ahmedabad,73.24,118.44,1.70,18.48,17.97,0.00,1.70,18.59,36.08,4.43,10.14,1.00,2015-01,Winter,142.784828,Moderate
2015-01-05,Ahmedabad,73.24,118.44,22.10,21.42,37.76,0.00,22.10,39.33,39.31,7.01,18.89,2.78,2015-01,Winter,330.289941,Very Poor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-27,Visakhapatnam,15.02,50.94,7.68,25.06,19.54,12.47,0.47,8.55,23.30,2.24,12.07,0.73,2020-06,Summer,43.107143,Good
2020-06-28,Visakhapatnam,24.38,74.09,3.42,26.06,16.53,11.99,0.52,12.72,30.14,0.74,2.21,0.38,2020-06,Summer,74.090000,Satisfactory
2020-06-29,Visakhapatnam,22.91,65.73,3.45,29.53,18.33,10.71,0.48,8.42,30.96,0.01,0.01,0.00,2020-06,Summer,65.730000,Satisfactory


In [ ]:
df_city_final.to_csv('/content/drive/MyDrive/impact_metrics/city_day_imp.csv', index = True)